# Remove the Glider Motion from the DVL Velocities
Without bottom tracking, the glider DVL measures the water velocity **relative to the glider**. To get the absolute water velocity, the glider's own motion has to be removed. This notebook:

1. Calibrates the steady-state Slocum glider flight model ([gliderflight](https://github.com/smerckel/gliderflight); Merckelbach et al., 2019) against the glider's depth rate, and checks the modeled glider velocity through the water against the velocity measured by the DVL in the cell closest to the glider
2. Constructs absolute velocity profiles for each dive by least squares from the DVL, the flight model, and the depth-averaged current (as for a lowered ADCP; Visbeck, 2002; Todd et al., 2017)
3. Checks the profiles and the glider's velocity over the ground against bottom tracking, which measures the glider's velocity over the ground directly and independently of the flight model and the depth-averaged current

Bottom tracking can't check the flight model on its own: the glider's velocity over the ground is its velocity through the water **plus the current at its depth**, and the only measurement of that current is the DVL itself.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from ooicgsn_glider_dvl import bottom_track, flight, inversion
from ooicgsn_glider_dvl.binning import bin_profiles
from ooicgsn_glider_dvl.merge import adcp_attitude_to_degrees
from ooicgsn_glider_dvl.utils import magnetic_declination

%matplotlib inline

In [ ]:
# CP05MOAS-GL379 deployment 8, deployed at the Pioneer NES array from 2019-09-27 to 2019-11-19
refdes = 'CP05MOAS-GL379-01-ADCPAM000'
eng_refdes = 'CP05MOAS-GL379-00-ENG000000'
deployment = 8
glider_name = '-'.join(refdes.split('-')[:2])
dep_str = f"{deployment:04d}"

# The glider's mass (kg), weighed before the deployment
glider_mass = 65.480

# Colors of the two sources compared in each figure
MODEL_COLOR, MEASURED_COLOR = '#2a78d6', '#eb6834'

### Load the Data
The merged dataset from notebook 2, the bottom-track data, and the glider engineering variables the flight model needs (pitch, heading, and buoyancy change). Datasets merged before the ADCP heading, pitch, and roll were converted to degrees are converted here (the conversion is skipped if they already are).

In [ ]:
glider = xr.open_dataset(f"../data/processed/{glider_name}.deployment{dep_str}.merged.nc").load()
glider = adcp_attitude_to_degrees(glider)

bt = xr.open_dataset(f"../data/raw/{refdes}.deployment{dep_str}.adcp_bottom_track.nc")

eng = pd.read_csv(
    f"../data/raw/deployment{dep_str}_{eng_refdes}-recovered_host-glider_eng_recovered.csv",
    usecols=flight.FLIGHT_VARIABLES,
)

### Magnetic Declination
OOI rotates the DVL water velocities from relative to magnetic north to relative to true north, but not the bottom-track velocities or the glider's heading. The rotation OOI applied is measured from the raw and corrected velocities, and used to rotate the others.

In [ ]:
declination = magnetic_declination(glider)
print(f"Magnetic declination: {declination:.2f} degrees")

### Calibrate the Flight Model
The glider's pitch, heading, and buoyancy change are interpolated to the DVL times. The model is calibrated against the depth rate during steady flight (deeper than 10 m, more than 5 m above the bottom of the dive, pitched more than 15 degrees, and not pumping), for the glider's volume (`Vg`) and drag coefficient (`Cd0`). It is calibrated for the whole deployment and for each week, to follow changes in drag from biofouling.

In [ ]:
glider = flight.add_flight_variables(glider, eng, declination)
steady = flight.calibration_mask(glider)
print(f"Steady flight: {steady.mean():.0%} of the measurements")

calibration = flight.calibrate_flight_model(glider, mass=glider_mass, window='7D')
calibration

In [ ]:
glider = flight.solve_flight_model(glider, calibration)

### Check the Flight Model Against the DVL
In the cell closest to the glider (about 3 m below it), the water moves with nearly the same current as at the glider, so the opposite of the DVL relative velocity is the glider's velocity through the water. The model was calibrated only against the depth rate, so this is an independent check.

In [ ]:
glider = flight.dvl_glider_velocity(glider)
flight.flight_model_skill(glider, steady).round(3)

In [ ]:
heading = np.deg2rad(glider['vehicle_heading'])
model_forward = (glider['glider_eastward_velocity_through_water'] * np.sin(heading)
                 + glider['glider_northward_velocity_through_water'] * np.cos(heading))
difference = (model_forward - glider['dvl_glider_forward_speed']).values[steady]
difference = difference[np.isfinite(difference)]

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(difference * 100, bins=np.arange(-30, 30.5, 1), color=MODEL_COLOR)
ax.axvline(0, color='0.3', linewidth=1)
ax.set_xlabel('Flight model minus DVL forward speed (cm s$^{-1}$)')
ax.set_ylabel('Measurements')
ax.set_title(f'{glider_name} deployment {deployment}: glider speed through the water')
ax.grid(alpha=0.3)

### Construct the Absolute Velocity Profiles
For each dive, the water velocity profile on 2 m depth bins and the glider's velocity over the ground in each ensemble are solved for together by least squares. The differences between the cells of each ensemble give the shear, the flight model links the ensembles to the glider's depth, and the depth-averaged current sets the absolute level.

Where there are only a few good cells per ensemble (e.g. in deep water with few scatterers), the cells split the profile into separate segments that only the flight model links, and errors in the flight model add up over depth. These profiles (`n_segments > 1`) are not solved.

In [ ]:
profiles, ground = inversion.invert_profiles(glider, steady=steady)

solved = profiles['eastward_seawater_velocity'].notnull().any('depth')
print(f"Solved {int(solved.sum())} of {profiles.sizes['profile']} profiles")
segments = profiles['n_segments']
print(f"Not solved: {int((segments == 0).sum())} without equations, {int((segments > 1).sum())} split into more than one segment")

### Bottom Tracking
The bottom-track velocities are rotated to true north and subtracted from the DVL water velocities, then binned on the same 2 m depth bins.

In [ ]:
glider = bottom_track.add_bottom_track(glider, bt, declination)
glider = bottom_track.remove_bottom_track(glider)

bt_profiles = bin_profiles(
    glider, velocity_variables=bottom_track.BOTTOM_TRACK_VELOCITY_VARIABLES, science_variables=()
)

### Check the Profiles Against Bottom Tracking
The difference of each profile is split into its depth-mean (`offset`), the error of the reference to the depth-averaged current, and the rest (`shape_rms`), the error of the shape of the profile.

In [ ]:
comparison = bottom_track.compare_profiles(profiles, bt_profiles)

summary = comparison[[
    'eastward_seawater_velocity_offset', 'northward_seawater_velocity_offset',
    'eastward_seawater_velocity_shape_rms', 'northward_seawater_velocity_shape_rms',
]].to_dataframe().dropna()
print(f"{len(summary)} profiles with both")
summary.abs().describe(percentiles=[0.5, 0.9]).round(3)

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(9, 6), sharey=True)
enough = comparison['n_profiles'] >= 10
for ax, component in zip(axes, ['eastward', 'northward']):
    bias = comparison[f'{component}_seawater_velocity_bias'].where(enough) * 100
    rms = comparison[f'{component}_seawater_velocity_rms'].where(enough) * 100
    ax.plot(bias, comparison['depth'], color=MODEL_COLOR, linewidth=2, label='Bias')
    ax.plot(rms, comparison['depth'], color=MEASURED_COLOR, linewidth=2, label='RMS')
    ax.axvline(0, color='0.3', linewidth=1)
    ax.set_xlabel(f'Inversion minus bottom track, {component} (cm s$^{{-1}}$)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Depth (m)')
axes[0].set_ylim(float(comparison['depth'].where(enough).max()) + 5, 0)
axes[0].legend()
fig.suptitle(f'{glider_name} deployment {deployment}: depth bins with at least 10 profiles')

##### Example Dive

In [ ]:
dive = 731
model_dive = profiles.swap_dims(profile='profile_id').sel(profile_id=dive)
bt_dive = bt_profiles.swap_dims(profile='profile_id').sel(profile_id=dive)

fig, axes = plt.subplots(ncols=2, figsize=(9, 6), sharey=True)
for ax, component in zip(axes, ['eastward', 'northward']):
    ax.plot(model_dive[f'{component}_seawater_velocity'] * 100, model_dive['depth'],
            color=MODEL_COLOR, linewidth=2, label='Flight model + depth-averaged current')
    ax.plot(bt_dive[f'bt_{component}_seawater_velocity_mean'] * 100, bt_dive['depth'],
            color=MEASURED_COLOR, linewidth=2, label='Bottom track')
    ax.set_xlabel(f'{component.capitalize()} velocity (cm s$^{{-1}}$)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Depth (m)')
deepest = max(
    float(model_dive['depth'].where(model_dive['n_cells'] > 0).max()),
    float(bt_dive['depth'].where(bt_dive['bt_eastward_seawater_velocity_count'] > 0).max()),
)
axes[0].set_ylim(deepest + 5, 0)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False)
fig.subplots_adjust(bottom=0.17)
fig.suptitle(f'{glider_name} deployment {deployment}: dive {dive}')

### Check the Glider's Velocity Over the Ground Against Bottom Tracking

In [ ]:
ground_comparison = bottom_track.compare_glider_velocity(glider, ground)
ground_comparison[['difference_eastward', 'difference_northward']].describe(percentiles=[0.1, 0.5, 0.9]).round(3)

### Save the Results

In [ ]:
profiles.to_netcdf(f"../data/processed/{glider_name}.deployment{dep_str}.velocity_profiles.nc")
bt_profiles.to_netcdf(f"../data/processed/{glider_name}.deployment{dep_str}.bottom_track_profiles.nc")
calibration.to_csv(f"../data/processed/{glider_name}.deployment{dep_str}.flight_calibration.csv", index=False)